# CIE 2 – Activity 1: Dataset Preparation

**Course:** Machine Learning (ADPC509)  
**Unit:** I – Foundations & Data Preparation  
**Marks:** 5

## 1. Dataset selection and source
This activity uses the **Diabetes dataset distributed with scikit-learn**. It contains 442 records and 10 numerical input measurements, with a continuous target representing a quantitative measure of disease progression one year after baseline. The dataset is publicly available through scikit-learn.

- Dataset documentation: https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset
- Dataset loader: `sklearn.datasets.load_diabetes`
- File type used in this notebook: CSV
- Note: the original dataset contains numerical features. To demonstrate mixed-type preparation, `bmi_category` is derived from BMI. A small number of missing values are **deliberately introduced in a working copy** to demonstrate missing-value handling; these are not claimed to be missing in the original source dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

# Load the public dataset
dataset = load_diabetes(as_frame=True)
df_original = dataset.frame.copy()

print("Original dataset shape:", df_original.shape)
display(df_original.head())
display(df_original.tail())

## 2. Create a working copy and a categorical feature

The source dataset contains 442 rows and 11 columns when the target is included. The `bmi_category` column is derived from the numerical BMI feature to demonstrate categorical analysis and encoding.

In [ ]:
df = df_original.copy()
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[-np.inf, -0.02, 0.02, np.inf],
    labels=["Low", "Medium", "High"]
)

# Introduce a few demonstration missing values in the working copy only.
df.loc[[5, 19, 44, 88, 120], "bmi"] = np.nan
df.loc[[12, 57, 101], "bmi_category"] = np.nan
df.loc[[30, 31], "bp"] = np.nan

print("Working dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data type"))
display(df.head())

## 3. Numerical and categorical columns; summary statistics

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
display(df[numeric_cols].describe().T)
for col in categorical_cols:
    print(f"\nFrequency counts for {col}:")
    display(df[col].value_counts(dropna=False))

## 4. Missing-value analysis

In [ ]:
missing_count = df.isna().sum()
missing_percent = (missing_count / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "Missing count": missing_count,
    "Missing percentage": missing_percent
})
display(missing_summary)

print("Columns without missing values:", missing_summary[missing_summary["Missing count"] == 0].index.tolist())
print("Columns requiring attention:", missing_summary[missing_summary["Missing count"] > 0].index.tolist())

## 5. Handle missing values

A copy is retained before cleaning. Median imputation is used for numerical columns because it is less affected by extreme values than the mean. Mode imputation is used for the categorical column. No rows or columns are deleted.

In [ ]:
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Missing values after imputation:")
display(df_clean.isna().sum().to_frame("Remaining missing values"))

## 6. Duplicate and consistency checks

In [ ]:
duplicate_count = df_clean.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

# Remove only exact duplicate rows, if any.
df_clean = df_clean.drop_duplicates().copy()

# Normalize categorical labels (trim whitespace and standardize capitalization).
for col in df_clean.select_dtypes(exclude=np.number).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

print("Shape after duplicate check:", df_clean.shape)

## 7. Exploratory Data Analysis (EDA)

The following visualizations examine the distribution of BMI and the frequency of the derived BMI categories.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_clean["bmi"], bins=20, edgecolor="black")
plt.title("Distribution of BMI")
plt.xlabel("BMI (standardized dataset value)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
df_clean["bmi_category"].value_counts().reindex(["Low", "Medium", "High"]).plot(kind="bar")
plt.title("Frequency of BMI Categories")
plt.xlabel("BMI Category")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

print("EDA observations:")
print("1. The BMI histogram shows the spread of BMI values in the dataset.")
print("2. The bar chart shows how records are distributed among the derived BMI categories.")
print("3. BMI category is derived from BMI; it is not an original source column.")

## 8. Encode categorical features

`bmi_category` is nominal for this demonstration, so one-hot encoding is used. The original category is represented by separate binary columns.

In [ ]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=["bmi_category"],
    drop_first=False,
    dtype=int
)
print("Columns after one-hot encoding:")
print(df_encoded.columns.tolist())
display(df_encoded.head())

## 9. Scale numerical features

Standardization is applied to numerical input features (mean approximately 0 and standard deviation approximately 1). The target column is not scaled, and no identifier column is scaled.

In [ ]:
feature_numeric_cols = [c for c in numeric_cols if c != "target"]
scaler = StandardScaler()
df_encoded[feature_numeric_cols] = scaler.fit_transform(df_encoded[feature_numeric_cols])

display(df_encoded[feature_numeric_cols].describe().T)

## 10. Save and verify the prepared dataset

In [ ]:
output_file = "prepared_dataset.csv"
df_encoded.to_csv(output_file, index=False)

print("Final shape:", df_encoded.shape)
print("Remaining missing values:", int(df_encoded.isna().sum().sum()))
print("Remaining duplicate rows:", int(df_encoded.duplicated().sum()))
print("Saved file:", output_file)
display(df_encoded.head())

## Conclusion

The Diabetes dataset was loaded and inspected. A categorical BMI group was derived to demonstrate categorical analysis. Missing values were introduced only in the working copy for practice and handled using median and mode imputation. Exact duplicate rows were checked, the categorical feature was one-hot encoded, and numerical input features were standardized while the target was left unchanged. The prepared dataset was saved as `prepared_dataset.csv`.

**Limitation:** the missing values in this exercise were introduced for demonstration and should not be interpreted as original missing values in the public dataset.

## GitHub publication checklist

Create a separate **public** repository, for example `ML-CIE2-Activity1-YourRollNumber`, and upload:
- This notebook (`Activity_1_Dataset_Preparation.ipynb`)
- `prepared_dataset.csv`
- `diabetes_eda_source_with_demo_missing.csv`
- A `README.md` describing the activity, dataset source and results

Add this acknowledgement to the README:

> This project was completed as part of the Machine Learning CIE 2 activity. Faculty reference and Machine Learning learning resources: https://github.com/Princy2129

Faculty profile: https://github.com/Princy2129